# שיעור 12 - צמצום היסטוריית שיחה עם לוח רישום סוכן

מחברת זו מדגימה כיצד לנהל הקשר בשיחות ארוכות באמצעות Microsoft Agent Framework. ככל שהשיחות מתארכות, מספר הטוקנים גדל — עד שגובה את החלון הקונטקסטואלי של המודל. אנו מטפלים בכך באמצעות **תבנית סיכום הקשר** ו-**לוח רישום סוכן** לזיכרון מתמשך.

## מה תלמדו:
1. **למה ניהול הקשר חשוב**: הבנת מגבלות טוקנים וחלונות הקשר
2. **סוכנים מודעי הקשר**: בניית סוכנים המנהלים את ההקשר של השיחה שלהם
3. **תבנית סיכום הקשר**: שימוש בכלים לדחיסת היסטוריית השיחה
4. **לוח רישום סוכן**: זיכרון מתמשך השורד צמצום הקשר

## דרישות מוקדמות:
- הגדרת Azure OpenAI עם משתני סביבה מוגדרים
- הבנה של מושגי סוכן בסיסיים משיעורים קודמים


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## מדוע ניהול הקשר חשוב

לכל מודל שפה גדול (LLM) יש **חלון הקשר** סופי — מספר הטוקנים המקסימלי שהוא יכול לעבד בבקשה אחת. ככל שהשיחה הרב-סיבובית מתקדמת:

- **מספר הטוקנים גדל בקו ישר** עם כל הודעת משתמש ותשובת העוזר.
- **טוקנים של ההנחיה שולטות בעלות** כי כל ההיסטוריה נשלחת מחדש בכל סיבוב.
- בסופו של דבר השיחה **עוברת את חלון ההקשר** והמודל או מקצר או מחזיר שגיאה.

### אסטרטגיות לניהול הקשר

| אסטרטגיה | איך זה עובד | היתכנות\קונפליקט |
|---|---|---|
| **קיצור** | זורק את ההודעות הישנות ביותר | מאבד הקשר מוקדם |
| **סיכום** | מדחס הודעות ישנות לסיכום | חלק מהפירוט אבוד, אך נקודות מפתח נשמרות |
| **מחברת / זיכרון חיצוני** | מאחסן עובדות חשובות מחוץ לשיחה | דורש קריאות לכלים, אך שורד כל קיצור |

במחברת זו אנו משלבם **סיכום** עם **כלי מחברת** כך שהסוכן יכול לשמור על רציפות גם כאשר היסטוריית השיחה מדוכאת.


## יצירת סוכן המודע להקשר


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## סימולציה של שיחה ארוכה

בואו נעבור דרך שיחה מרובת סיבובים כדי לראות איך ההקשר מצטבר. הסוכן צריך לשמור פרטים מרכזיים (העדפות, תקציב, תאריכי נסיעה) לאורך הסיבובים ולהדגים רצף.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

שימו לב כיצד הסוכן שומר על ההקשר מהפעמים הקודמות — הוא יודע על יפן, סושי, מקדשים, צילום, התקציב של 3000$, נסיעה לבד, וטווח הזמן של אפריל. בשיחה קצרה זה עובד טוב, אבל ככל שהשיחה מתארכת ההיסטוריה המלאה הופכת ליקרה לשליחה מחדש.

בואו נמשיך את השיחה עם עוד סבבים כדי לראות את הצטברות ההקשר:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## תבנית סיכום הקשר

עם התרחבות השיחה, נוכל להשתמש ב**כלי סיכום** כדי לדחוס את ההקשר שהצטבר לפורמט תמציתי. הסוכן קורא לכלי זה כדי לתעד העדפות מרכזיות כך שגם אם הודעות ישנות יימחקו, המידע החיוני יישמר.

תבנית זו היא אבני הבניין לצמצום היסטוריה מתוחכם יותר:
1. הסוכן מזהה עובדות מרכזיות מהשיחה
2. הוא קורא לכלי הסיכום כדי לשמור אותן
3. הודעות ישנות יכולות להימחק בבטחה כי הסיכום תופס את מה שחשוב

להלן אנו מגדירים את כלי `summarize_preferences` שהסוכן יכול לקרוא לו כדי לתעד סיכום תמציתי של מה שלמד.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## סיכום

בשיעור זה למדת כיצד לנהל הקשר בשיחות סוכנים ארוכות-טווח באמצעות Microsoft Agent Framework:

### מושגים מרכזיים
- **חלונות הקשר הם סופיים** — כל טוקן בהיסטוריית השיחה עולה כסף ונחשב למגבלה.
- **כלי סיכום** מאפשרים לסוכן לדחוס את ההקשר המצטבר לתמצית קומפקטית, מה שמפחית את שימוש הטוקנים תוך שמירה על המידע החיוני.
- **מחברות סוכן** מספקות זיכרון חיצוני מתמשך ששרד כל הקטנת שיחה.

### מה שבנית
- **סוכן מודע להקשר** ששומר על רצף בשיחות רב-סבביות
- **כלי סיכום** (`summarize_preferences`) שמתעד פרטים מרכזיים של המשתמש בפורמט תמציתי
- **שיחה רב-סבבית** המדגימה שמירת הקשר והתמודדות עם שינויים

### יישומים בעולם האמיתי
- **בוטי שירות לקוחות**: זוכרים העדפות לאורך סשנים ארוכים של תמיכה
- **עוזרים אישיים**: עוקבים אחרי פרויקטים מתמשכים בלי צורך להסביר מחדש את ההקשר
- **מדריכי חינוך**: שומרים על התקדמות תלמיד לאורך אינטראקציות רבות

### צעדים הבאים
- מימוש כלי מחברת מלא עם אחסון מבוסס קבצים
- הוספת קיצוץ אוטומטי של ההיסטוריה לאחר הסיכום
- שילוב עם מסדי נתונים וקטוריים לחיפוש זיכרון סמנטי
- בניית סוכנים שיכולים להמשיך שיחות ימים לאחר מכן עם כל ההקשר


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש להביא בחשבון כי תרגומים אוטומטיים עלולים להכיל טעויות או אי-דיוקים. המסמך המקורי בשפת המקור שלו הוא המקור הסמכותי. למידע קריטי מומלץ לפנות לתרגום מקצועי על ידי מתרגם אנושי. אנו לא נושאים באחריות על אי-הבנות או פרשנויות שגויות הנובעות משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
